# 自定义层

深度学习成功背后的一个因素是神经网络的灵活性：
我们可以用创造性的方式组合不同的层，从而设计出适用于各种任务的架构。
例如，研究人员发明了专门用于处理图像、文本、序列数据和执行动态规划的层。
有时我们会遇到或要自己发明一个现在在深度学习框架中还不存在的层。
在这些情况下，必须构建自定义层。本节将展示如何构建自定义层。

## 不带参数的层

首先，我们(**构造一个没有任何参数的自定义层**)。
回忆一下在 :numref:`sec_model_construction`对块的介绍，
这应该看起来很眼熟。
下面的`CenteredLayer`类要从其输入中减去均值。
要构建它，我们只需继承基础层类并实现前向传播功能。


In [1]:
import torch
import torch.nn.functional as F
from torch import nn

# 自定义中心化层：使输出数据的均值为0
class CenteredLayer(nn.Module):
    def __init__(self):
        super().__init__()  # 初始化父类nn.Module

    def forward(self, X):
        return X - X.mean()  # 每个元素减去整体均值

让我们向该层提供一些数据，验证它是否能按预期工作。


In [2]:
# 实例化自定义中心化层
layer = CenteredLayer()

# 创建浮点型张量，并传入中心化层进行前向传播
layer(torch.FloatTensor([1, 2, 3, 4, 5]))

tensor([-2., -1.,  0.,  1.,  2.])

现在，我们可以[**将层作为组件合并到更复杂的模型中**]。


In [3]:
# 构建神经网络：全连接层 + 自定义中心化层
net = nn.Sequential(
    nn.Linear(8, 128),  # 全连接层：输入8维，输出128维
    CenteredLayer()     # 中心化层：使输出数据的整体均值为0
)

作为额外的健全性检查，我们可以在向该网络发送随机数据后，检查均值是否为0。
由于我们处理的是浮点数，因为存储精度的原因，我们仍然可能会看到一个非常小的非零数。


In [4]:
# 生成4个样本，每个样本包含8个特征，并进行前向传播
Y = net(torch.rand(4, 8))

# 计算网络输出的平均值
Y.mean()

tensor(7.4506e-09, grad_fn=<MeanBackward0>)

## 带参数的层

除了无参数的自定义层，我们还可以定义包含可训练参数的神经网络层。

### 1. 核心概念

- 权重（Weight）：控制输入特征对输出的影响。
- 偏置（Bias）：调整神经元的输出。
- 可训练参数：通过反向传播和优化算法不断更新的参数。

PyTorch 提供了参数管理机制，支持参数的初始化、访问、共享、保存和加载，无需手动实现。

### 2. 自定义全连接层

全连接层包含权重 \\(W\\) 和偏置 \\(b\\)，其计算公式为：

\\[ Y=\operatorname{ReLU}(XW+b) \\]

其中：

- `in_units`：输入特征数量。
- `units`：输出神经元数量。
- `ReLU`：激活函数，为网络引入非线性。

### 3. 核心总结

带参数的自定义层 = 可训练权重 + 偏置 + 前向传播计算。

与之前的 `CenteredLayer` 不同，带参数的层能够在训练过程中自动更新权重和偏置。

In [5]:
# 自定义带有可训练参数的全连接层
class MyLinear(nn.Module):
    def __init__(self, in_units, units):
        super().__init__()  # 初始化父类

        # 创建可训练的权重矩阵，形状为(输入维度, 输出维度)
        self.weight = nn.Parameter(torch.randn(in_units, units))

        # 创建可训练的偏置向量，长度为输出维度
        self.bias = nn.Parameter(torch.randn(units,))

    def forward(self, X):
        # 矩阵乘法并加上偏置，完成线性变换
        linear = torch.matmul(X, self.weight.data) + self.bias.data

        # 使用ReLU激活函数
        return F.relu(linear)

接下来，我们实例化`MyLinear`类并访问其模型参数。


In [6]:
# 创建自定义全连接层：输入5维，输出3维
linear = MyLinear(5, 3)

# 查看该层的权重参数
linear.weight

Parameter containing:
tensor([[ 1.1242, -0.4255, -0.1505],
        [ 2.1595, -0.2723, -0.4410],
        [-2.2348, -0.6786,  1.0236],
        [ 1.2724, -0.4784, -0.5722],
        [ 1.4788,  0.2639, -0.8358]], requires_grad=True)

我们可以[**使用自定义层直接执行前向传播计算**]。


In [7]:
# 生成2个样本，每个样本包含5个特征，并传入自定义全连接层
linear(torch.rand(2, 5))

tensor([[3.1718, 0.0000, 0.0000],
        [1.1144, 0.0000, 0.0000]])

我们还可以(**使用自定义层构建模型**)，就像使用内置的全连接层一样使用自定义层。


In [8]:
# 使用两个自定义全连接层构建神经网络
net = nn.Sequential(
    MyLinear(64, 8),  # 第一层：输入64维，输出8维
    MyLinear(8, 1)    # 第二层：输入8维，输出1维
)

# 生成2个样本，每个样本64个特征，并进行前向传播
net(torch.rand(2, 64))

tensor([[7.3302],
        [6.1877]])

## 小结

* 我们可以通过基本层类设计自定义层。这允许我们定义灵活的新层，其行为与深度学习框架中的任何现有层不同。
* 在自定义层定义完成后，我们就可以在任意环境和网络架构中调用该自定义层。
* 层可以有局部参数，这些参数可以通过内置函数创建。